# Lab Day 1 — Xây dựng mạng nơ-ron và thí nghiệm huấn luyện

**Hoàng Quốc Việt — 2A202602563** · Track 4 · Ngày 1 · VinUniversity AICB 2026

Notebook này chạy trên **Kaggle (GPU)**. Ô đầu tiên tự nhận biết môi trường:
- trên Kaggle: clone repo nộp bài (fork của `VinUni-AI20k/K4-Track4-Day1-Neuralnetwork`) vào `/kaggle/working/repo`, rồi làm việc tại `submission_2A202602563/code/`;
- chạy tại chỗ: làm việc ngay trong `submission_2A202602563/code/`.

Trong cả hai trường hợp `REPO_ROOT = "../.."` (chứa `data/`, `scripts/`, `templates/`) và `OUT_DIR = ".."` (thư mục nộp: `figures/`, `results/`, `experiments.xlsx`, `predictions_eval.csv`).

Quy ước: mọi lựa chọn (lr, epoch, cấu hình cuối) làm **chỉ bằng val**. Tập eval chỉ xuất hiện ở Part 4, cho baseline và cấu hình cuối cùng.

In [ ]:
# ===== Cấu hình đường dẫn và môi trường =====
import os, sys, json, time, math, subprocess
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from IPython.display import Image, display

REPO_URL = "https://github.com/Catnip-harvest/K4-Track4-Day1-HoangQuocViet-2A202602563-Neuralnetwork.git"
BRANCH = "feat/day1-lab"
SUB_DIR = "submission_2A202602563"

IN_KAGGLE = os.path.isdir("/kaggle/working")
if not os.path.exists("data.py"):              # notebook không nằm cạnh các module -> lấy repo về
    work = "/kaggle/working/repo" if IN_KAGGLE else os.path.abspath("repo")
    if not os.path.isdir(work):
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, work], check=True)
    os.chdir(os.path.join(work, SUB_DIR, "code"))
sys.path.insert(0, os.getcwd())

REPO_ROOT = "../.."     # gốc repo (data/, scripts/, templates/), tính từ submission_<MSSV>/code/
OUT_DIR = ".."          # thư mục nộp submission_<MSSV>/
print("cwd       :", os.getcwd())
print("REPO_ROOT :", os.path.abspath(REPO_ROOT))
print("OUT_DIR   :", os.path.abspath(OUT_DIR))

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("torch", torch.__version__, "| device", device, "| Kaggle" if IN_KAGGLE else "| local")
if device == "cuda":
    cap = torch.cuda.get_device_capability(0)
    BF16_NATIVE = cap >= (8, 0)                 # BF16 có phần cứng từ Ampere (8.x) trở lên
    print("GPU:", torch.cuda.get_device_name(0), "| compute capability", cap,
          "| BF16 phần cứng:", BF16_NATIVE, "| torch.cuda.is_bf16_supported():", torch.cuda.is_bf16_supported())
for d in ("figures", "results"):
    os.makedirs(f"{OUT_DIR}/{d}", exist_ok=True)

from data import prepare_data, iterate_batches, N_NUMERIC
from model import MLP, EXPECTED_PARAMS, count_params, init_weights, activation_stats
from optimizer import build_optimizer, clip_gradients
from train import (DEFAULT_CFG, set_seed, evaluate, predict, run_experiment, final_eval,
                   compute_loss, build_model)
from plots import plot_run, plot_compare
from results_table import save_result, load_results, to_row, write_xlsx

## Part 0 — Dữ liệu
Chia sẵn bằng metadata: `scripts/split_data.py` (script tự kiểm tra số dòng, `row_id`, số mẫu từng tập). Sau đó tách validation **từ train** (20%, phân tầng theo nhãn, seed 42), chuẩn hoá 10 cột số bằng mean/std **của phần train còn lại**, đưa mọi tensor lên GPU một lần.

In [ ]:
proc = subprocess.run([sys.executable, "scripts/split_data.py"], cwd=REPO_ROOT,
                      capture_output=True, text=True, encoding="utf-8", check=True)
print(proc.stdout)

data = prepare_data(device, val_fraction=0.2, seed=42, processed_dir=f"{REPO_ROOT}/data/processed")
X_tr, y_tr, X_val, y_val = data["X_tr"], data["y_tr"], data["X_val"], data["y_val"]
assert len(X_tr) == 371_847 and len(X_val) == 92_962 and len(data["X_eval"]) == 116_203

# tỉ lệ lớp ở 3 tập (phân tầng -> gần như bằng nhau)
ratios = pd.DataFrame({name: torch.bincount(t, minlength=7).cpu().numpy() / len(t) * 100
                       for name, t in [("train%", y_tr), ("val%", y_val), ("eval%", data["y_eval"])]})
print(ratios.round(3).to_string())

# mean/std của 10 cột số trên X_tr sau chuẩn hoá: phải ≈ 0 / 1; cột nhị phân giữ nguyên 0/1
m = X_tr[:, :N_NUMERIC].mean(0).cpu().numpy(); s = X_tr[:, :N_NUMERIC].std(0).cpu().numpy()
print("mean 10 cột số (X_tr):", np.round(m, 4)); print("std  10 cột số (X_tr):", np.round(s, 4))
assert np.abs(m).max() < 1e-3 and np.abs(s - 1).max() < 1e-3
assert set(torch.unique(X_tr[:, N_NUMERIC:]).tolist()) <= {0.0, 1.0}
print("val (không fit, chỉ áp dụng) — mean:", np.round(X_val[:, :N_NUMERIC].mean(0).cpu().numpy(), 3))

# eval_row_id vẫn khớp thứ tự mẫu của eval.npz (prepare_data không xáo eval)
ev = np.load(f"{REPO_ROOT}/data/processed/eval.npz")
assert np.array_equal(data["eval_row_id"], ev["row_id"])
assert np.array_equal(data["y_eval"].cpu().numpy(), ev["y"])
print("eval_row_id: khớp eval.npz,", len(data["eval_row_id"]), "dòng, không trùng")

n_full, rem = divmod(len(X_tr), 512)
print(f"batch 512: {n_full} lô đầy + 1 lô cuối {rem} mẫu -> giữ lô cuối, {n_full + 1} bước/epoch")

**Nhận xét Part 0.** Kích thước đúng như GUIDE (371 847 / 92 962 / 116 203), `X` là `float32` 54 cột, `y` là `int64` 0..6, tỉ lệ lớp ba tập gần như trùng nhau nhờ phân tầng. Sau chuẩn hoá, 10 cột số trên `X_tr` có mean ≈ 0, std ≈ 1; trên val chỉ *gần* 0 vì val được biến đổi bằng thống kê của train chứ không phải của chính nó — đúng ý đồ.

*Vì sao val và eval không tham gia tính mean/std:* phép chuẩn hoá là một phần của mô hình. Nếu tính mean/std trên val/eval thì thông tin về phân phối của dữ liệu dùng để chấm đã "rò" vào mô hình trước khi chấm, nên điểm val/eval không còn ước lượng trung thực cho dữ liệu chưa thấy. Áp dụng *cùng* mean/std của train lên eval thì được (đó là cách mô hình sẽ dùng ngoài thực tế), còn *tính* thống kê từ eval thì không.

*Lô cuối:* giữ lại lô nhỏ (135 mẫu với batch 512). Loss là trung bình trong lô nên thang gradient không đổi; lô nhỏ chỉ nhiễu hơn một chút. Mốc "luôn đoán lớp đa số" trên val ≈ 0,4876 accuracy.

## Part 1 — Model và kiểm tra "sức khoẻ" ban đầu
Quy định: `M-base` = `54 → 256 → 128 → 7`, **47 879 tham số** = (54·256+256) + (256·128+128) + (128·7+7), logits `(B, 7)`, không softmax trong model, khởi tạo He (`kaiming_normal_`, Var[W] = 2/n_in, bias = 0).

In [ ]:
set_seed(0)
model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
assert count_params(model) == EXPECTED_PARAMS[(256, 128)] == 47_879
print(model)
for name, p in model.named_parameters():
    print(f"  {name:14s} {tuple(p.shape)}  {p.numel()}")
print("tổng tham số:", count_params(model))

# He thật sự được áp dụng: std(W) ≈ sqrt(2/n_in), bias = 0
for m_ in model.net:
    if isinstance(m_, torch.nn.Linear):
        print(f"  Linear{tuple(m_.weight.shape)}: std(W) = {m_.weight.std().item():.4f}  "
              f"(He kỳ vọng {math.sqrt(2 / m_.in_features):.4f}), |b|max = {m_.bias.abs().max().item():.1f}")

# một lô (8, 54) đi qua từng lớp
h = torch.randn(8, 54, device=device)
print("input", tuple(h.shape))
for layer in model.net:
    h = layer(h); print(f"  sau {layer.__class__.__name__:7s} -> {tuple(h.shape)}")
assert h.shape == (8, 7)

In [ ]:
# Loss bước 0 trên TOÀN BỘ val, chế độ eval(), trước mọi bước cập nhật
step0 = evaluate(model, X_val, y_val, "ce")
print(f"loss bước 0 (val) = {step0['loss']:.4f}   | ln 7 = {math.log(7):.4f}   | lệch = {step0['loss'] - math.log(7):+.4f}")
print("std của logits ở bước 0:", round(activation_stats(model, X_val[:4096])[-1], 3))

In [ ]:
# Quá khớp 20 mẫu: dropout = 0, cùng code đường đi (MLP + compute_loss + build_optimizer), 500 bước
set_seed(0)
tiny = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
idx = torch.randperm(len(X_tr), generator=torch.Generator().manual_seed(0))[:20].to(device)
x20, y20 = X_tr[idx], y_tr[idx]
print("nhãn của 20 mẫu:", y20.tolist())
opt = build_optimizer("adam", tiny.parameters(), lr=1e-3)
curve = []
for step in range(500):
    tiny.train()
    loss = compute_loss(tiny(x20), y20, "ce")
    opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
    curve.append(loss.item())
tiny.eval()
with torch.no_grad():
    acc20 = (tiny(x20).argmax(1) == y20).float().mean().item()
print(f"loss bước 0 = {curve[0]:.4f} | bước 100 = {curve[99]:.4f} | bước 500 = {curve[-1]:.6f} | acc trên 20 mẫu = {acc20:.2f}")
assert curve[-1] < 0.01 and acc20 == 1.0

plt.figure(figsize=(6, 3.5))
plt.semilogy(curve); plt.xlabel("bước"); plt.ylabel("CE loss (log)")
plt.title("Quá khớp 20 mẫu — M-base, He, Adam lr=1e-3, dropout 0"); plt.grid(alpha=0.3); plt.show()

In [ ]:
# Gradient có chảy tới mọi tham số không? (một lô val 512 mẫu, model vừa khởi tạo)
set_seed(0)
gm = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
gm.train(); gm.zero_grad()
compute_loss(gm(X_val[:512]), y_val[:512], "ce").backward()
names = ["W1", "b1", "W2", "b2", "W3", "b3"]
for nm, (pname, p) in zip(names, gm.named_parameters()):
    assert p.grad is not None, f"{pname} không có gradient"
    gnorm = p.grad.norm().item()
    print(f"  {nm} ({pname:12s}) ‖grad‖ = {gnorm:.5f}")
    assert gnorm > 0, f"{pname} có gradient bằng 0"
print("mọi tham số có gradient khác None và khác 0")

**Nhận xét Part 1** — xem các số in ở trên; tóm tắt và so sánh nằm trong `REPORT.md` mục 2.
- Loss bước 0 lệch khỏi ln 7 một chút. ln 7 là loss khi logits của 7 lớp **bằng nhau** (softmax đều 1/7). Với He, mỗi lớp giữ phương sai tín hiệu ≈ không đổi, nên logits ở bước 0 có độ lệch chuẩn cỡ 1 chứ không phải 0 → softmax đã hơi "tự tin" một cách ngẫu nhiên, và loss trung bình cao hơn ln 7 một ít. Lệch nhỏ (không phải gấp nhiều lần) nên chuẩn hoá đầu vào và độ lớn lớp cuối là hợp lý.
- Quá khớp 20 mẫu: loss về ~0 và acc 100% → model, loss, `zero_grad`, danh sách tham số của optimizer phối hợp đúng. Đây là bằng chứng code học được, không phải bằng chứng tổng quát hoá.
- Mọi `W` và `b` đều nhận gradient khác 0 sau `backward()`.

## Part 2 — Pipeline và baseline
`run_experiment(cfg, data)` là hàm huấn luyện **duy nhất**; mọi thí nghiệm chỉ đổi dict `cfg`. Mỗi epoch ghi: train loss (đo ở `eval()` trên một tập con cố định 50 000 mẫu train), val loss / acc / macro-F1, `grad_norm` trung bình và max (chuẩn L2 toàn cục **trước** clip), tỉ lệ bước bị clip, thời gian huấn luyện (có `torch.cuda.synchronize()`). Tóm tắt: loss bước 0, best epoch theo **val loss**, metric tại best epoch, thời gian/epoch, bộ nhớ GPU cực đại (phần vượt trên dữ liệu đã nằm sẵn trên GPU), cờ `diverged`. `best_state` = bản sao trọng số tại best epoch, dùng ở Part 4.

**Chọn lr cho baseline (SGD + momentum 0,9, batch 512, 20 epoch, seed 1) bằng val.** Thử 6 giá trị cách nhau ~3 lần, 0,003 → 1. (Lần chạy thăm dò đầu tiên chỉ đến 0,3 và 0,3 thắng — nằm ở biên lưới, nên lưới được nới tới 1 để chắc lr tốt nhất nằm *bên trong* khoảng đã thử.)
*Dự đoán:* lr nhỏ (0,003) học chậm, chưa hội tụ sau 20 epoch; lr lớn hơn tốt dần vì best epoch đều là epoch cuối (chưa hội tụ); 0,3 với momentum 0,9 có bước hiệu dụng ≈ η/(1−μ) = 3; 1,0 (bước hiệu dụng 10) có thể dao động hoặc phân kỳ. Đoán lr tốt nhất là 0,1 hoặc 0,3.

In [ ]:
RESULTS = {}

def run(cfg, show=False):
    # Chạy một cấu hình, lưu ảnh figures/<exp_id>.png và results/<exp_id>.json ngay sau khi chạy.
    if "QUICK_EPOCHS" in globals():
        cfg = {**cfg, "epochs": min(cfg["epochs"], QUICK_EPOCHS)}
    res = run_experiment(cfg, data, verbose=False)
    eid = res["cfg"]["exp_id"]
    save_result(res, f"{OUT_DIR}/results")
    plot_run(res, f"{OUT_DIR}/figures/{eid}.png")
    RESULTS[eid] = res
    s = res["summary"]
    if s["best_epoch"] is None:
        print(f"{eid:24s} DIVERGED ở epoch {s['diverged_epoch']} (step0 {s['step0_loss']:.3f})")
    else:
        print(f"{eid:24s} step0 {s['step0_loss']:.3f} | best ep {s['best_epoch']:2d} | val_loss {s['best_val_loss']:.4f} "
              f"| acc {s['val_acc']:.4f} | F1 {s['val_macro_f1']:.4f} | train {s['final_train_loss']:.4f} "
              f"| gn {res['history']['grad_norm'][-1]:.3f} | {s['time_per_epoch_s']:.2f}s/ep"
              + (" | DIVERGED" if s["diverged"] else ""))
    if show:
        display(Image(f"{OUT_DIR}/figures/{eid}.png"))
    return res

LR_GRID = [0.003, 0.01, 0.03, 0.1, 0.3, 1.0]
sweep = []
for lr in LR_GRID:
    sweep.append(run({**DEFAULT_CFG, "exp_id": f"lr-sgdm-{lr:g}", "group": "hparam",
                      "description": f"Tìm lr baseline: SGD+momentum lr={lr:g} (còn lại như baseline)",
                      "lr": lr, "seed": 1,
                      "notes": "Dò lr cho baseline bằng val. Dự đoán: lr nhỏ chưa hội tụ, 0.1-0.3 tốt nhất, 1.0 có thể dao động/phân kỳ"}))

ok = [r for r in sweep if not r["summary"]["diverged"]]
best = max(ok, key=lambda r: r["summary"]["val_macro_f1"])
BASE_LR = best["cfg"]["lr"]
print(f"\nlr chọn bằng val macro-F1: {BASE_LR:g}  ({best['cfg']['exp_id']}, F1 {best['summary']['val_macro_f1']:.4f})")
plot_compare(sweep, ["val_loss", "val_macro_f1", "grad_norm"], f"{OUT_DIR}/figures/compare_lr_sgdm.png",
             "Dò lr cho baseline (SGD+momentum 0.9, batch 512, 20 epoch, seed 1)")
display(Image(f"{OUT_DIR}/figures/compare_lr_sgdm.png"))

**Baseline với 3 seed** (seed huấn luyện 1, 2, 3; tách val vẫn seed 42). `base-s1` chạy lại đúng cấu hình của dòng dò lr đã chọn — cùng seed nên kỳ vọng ra (gần như) cùng số, đây cũng là phép kiểm tra tính lặp lại.

In [ ]:
BASE_CFG = {**DEFAULT_CFG, "lr": BASE_LR}
base_runs = []
for s in (1, 2, 3):
    base_runs.append(run({**BASE_CFG, "exp_id": f"base-s{s}", "group": "baseline", "seed": s,
                          "description": f"Baseline M-base, SGD+momentum lr={BASE_LR:g}, seed {s}",
                          "notes": "Baseline bắt buộc; lr chọn bằng val ở lr-sgdm-*"}, show=(s == 1)))

f1s = np.array([r["summary"]["val_macro_f1"] for r in base_runs])
accs = np.array([r["summary"]["val_acc"] for r in base_runs])
vls = np.array([r["summary"]["best_val_loss"] for r in base_runs])
NOISE = dict(f1_mean=f1s.mean(), f1_std=f1s.std(ddof=1), acc_mean=accs.mean(), acc_std=accs.std(ddof=1),
             loss_mean=vls.mean(), loss_std=vls.std(ddof=1))
TWO_SIGMA = 2 * NOISE["f1_std"]
print(f"val macro-F1 = {NOISE['f1_mean']:.4f} ± {NOISE['f1_std']:.4f} (σ mẫu, 3 seed) -> ngưỡng nhiễu 2σ = {TWO_SIGMA:.4f}")
print(f"val acc      = {NOISE['acc_mean']:.4f} ± {NOISE['acc_std']:.4f}")
print(f"best val loss= {NOISE['loss_mean']:.4f} ± {NOISE['loss_std']:.4f}")
print("best epoch các seed:", [r["summary"]["best_epoch"] for r in base_runs])
print(f"lặp lại: lr-sgdm-{BASE_LR:g} F1 {best['summary']['val_macro_f1']:.6f} vs base-s1 F1 {base_runs[0]['summary']['val_macro_f1']:.6f}")
plot_compare(base_runs, ["train_loss", "val_loss", "val_macro_f1"], f"{OUT_DIR}/figures/compare_baseline.png",
             f"Baseline 3 seed (SGD+momentum lr={BASE_LR:g})")
display(Image(f"{OUT_DIR}/figures/compare_baseline.png"))

def verdict(eid, ref=None):
    # Chênh lệch val macro-F1 so với trung bình baseline (hoặc một lần chạy tham chiếu), so với 2σ seed.
    s = RESULTS[eid]["summary"]
    if s["val_macro_f1"] is None:
        return f"{eid}: không có metric (diverged ngay)"
    ref_f1 = NOISE["f1_mean"] if ref is None else RESULTS[ref]["summary"]["val_macro_f1"]
    d = s["val_macro_f1"] - ref_f1
    tag = "VƯỢT nhiễu" if abs(d) > TWO_SIGMA else "trong nhiễu"
    return f"{eid}: F1 {s['val_macro_f1']:.4f}, Δ = {d:+.4f} so với {'TB baseline' if ref is None else ref} ({tag}, 2σ = {TWO_SIGMA:.4f})"

**Nhận xét baseline** — số cụ thể ở output trên và trong `REPORT.md` mục 2. Hình dạng đường cong và độ nhiễu được dùng làm thước đo cho mọi so sánh ở Part 3: một chênh lệch val macro-F1 nhỏ hơn 2σ không được coi là bằng chứng.

## Part 3 — Thí nghiệm
Mỗi thí nghiệm: dự đoán viết **trước khi chạy** → đổi **một** yếu tố so với `BASE_CFG` (seed 1, cùng split, 20 epoch trừ khi chính epoch là yếu tố) → chạy, lưu ảnh + JSON → đối chiếu. Lời dự đoán cũng được ghi vào cột `notes` của bảng. Chênh lệch được so với trung bình 3 seed baseline và ngưỡng 2σ.

In [ ]:
def exp(exp_id, group, description, notes, **changes):
    return run({**BASE_CFG, "exp_id": exp_id, "group": group, "description": description,
                "notes": "Dự đoán: " + notes, **changes})

### Chủ đề 1 — Hàm mất mát: CE vs MSE (`loss-mse-*`)
MSE ở đây: `mean over B·7 phần tử of (z − onehot(y))²`, không hệ số 1/2 (giống `nn.MSELoss`), tính trên **logit thô**.

**Dự đoán (trước khi chạy).** Gradient theo logit: CE cho `(softmax(z) − onehot)/B` (mỗi phần tử ∈ [−1, 1], lớn nhất khi sai nặng), MSE cho `2(z − onehot)/(7B)` — nhỏ hơn ~3,5 lần ở cùng sai số và kéo mọi logit về đúng 0 hoặc 1. MSE vì vậy học chậm hơn ở cùng lr; hơn nữa hồi quy về one-hot trên dữ liệu mất cân bằng đẩy logit của lớp hiếm về gần 0, nên lớp hiếm ít khi thắng argmax → **macro-F1 của MSE thấp hơn CE rõ rệt (vượt 2σ), accuracy giảm ít hơn**. Cho MSE lr gấp 3 (`loss-mse-lr…`) bù một phần thang gradient nhỏ nhưng không bù được vấn đề lớp hiếm. Không so độ lớn loss (khác thang đo); so acc, macro-F1, tốc độ hội tụ.

In [ ]:
loss_runs = [exp("loss-mse", "loss", f"MSE trên logit (one-hot), lr={BASE_LR:g} như baseline",
                 "MSE học chậm hơn, macro-F1 thấp hơn CE (lớp hiếm bị bỏ qua)", loss="mse"),
             exp(f"loss-mse-lr{3*BASE_LR:g}", "loss", f"MSE trên logit, lr={3*BASE_LR:g} (gấp 3 để bù gradient nhỏ)",
                 "lr gấp 3 cải thiện MSE nhưng vẫn thua CE ở macro-F1", loss="mse", lr=3 * BASE_LR)]
for r in loss_runs:
    print(verdict(r["cfg"]["exp_id"]))
plot_compare([RESULTS["base-s1"]] + loss_runs, ["val_acc", "val_macro_f1", "grad_norm"],
             f"{OUT_DIR}/figures/compare_loss.png", "CE (base-s1) vs MSE — so bằng acc/macro-F1, không so loss")
display(Image(f"{OUT_DIR}/figures/compare_loss.png"))

**Đối chiếu (CE vs MSE):** xem `REPORT.md` mục 3.1.

### Chủ đề 2 — Bộ tối ưu hoá (`opt-*`)
Mỗi bộ thử 3–4 lr cách nhau ~3 lần; so ở lr tốt nhất của mỗi bộ (theo val macro-F1). Lưới đã được nới sau lần thăm dò đầu (lr tốt nhất của SGD, Adam, AdamW khi đó đều nằm ở biên lưới), để mỗi "lr tốt nhất" có hàng xóm ở cả hai phía hoặc ghi rõ là ở biên. SGD+momentum dùng chính các lần dò lr ở Part 2 (`lr-sgdm-*`). Tham số: SGD+momentum `μ = 0,9`; Adam/AdamW `betas = (0,9; 0,999)`, `eps = 1e-8`; AdamW `weight_decay = 0,01`.

**Dự đoán (trước khi chạy).**
- SGD thuần (không momentum) cần lr lớn hơn ~10 lần để đi được quãng đường như SGD+momentum (bước hiệu dụng của momentum ≈ η/(1−μ) = 10η). Ở cùng lr thì kém hẳn; ở lr tốt nhất của nó (0,3–1) đến gần SGD+momentum.
- Adam tốt nhất quanh 1e-3; xuống nhanh hơn ở các epoch đầu vì mỗi tham số được chia bước theo `√v̂` của chính nó. Ở lr tốt nhất của mỗi bộ, khoảng cách Adam vs SGD+momentum sẽ nhỏ hơn nhiều so với khi so ở một lr chung — có thể vẫn vượt 2σ vì cả hai chưa hội tụ sau 20 epoch.
- AdamW (wd 0,01) ≈ Adam: suy giảm `η·λ·w` mỗi bước rất nhỏ trong 20 epoch, mô hình lại chưa quá khớp nên không có gì để chính quy hoá.
- AdamW với `weight_decay = 0` phải cho **cùng** kết quả với Adam cùng lr (khác biệt chỉ do sai số dấu phẩy động).

In [ ]:
opt_runs = {"sgd": [], "adam": [], "adamw": []}
for lr in (0.03, 0.1, 0.3, 1.0, 3.0):
    opt_runs["sgd"].append(exp(f"opt-sgd-lr{lr:g}", "optimizer", f"SGD không momentum, lr={lr:g}",
                               "SGD thuần cần lr ~10x SGD+momentum; ở lr nhỏ học chậm", optimizer="sgd", lr=lr))
for lr in (3e-4, 1e-3, 3e-3, 1e-2):
    opt_runs["adam"].append(exp(f"opt-adam-lr{lr:g}", "optimizer", f"Adam, lr={lr:g}",
                                "Adam tốt nhất quanh 1e-3, hội tụ nhanh ở epoch đầu", optimizer="adam", lr=lr))
for lr in (1e-3, 3e-3, 1e-2):
    opt_runs["adamw"].append(exp(f"opt-adamw-lr{lr:g}", "optimizer", f"AdamW wd=0.01, lr={lr:g}",
                                 "AdamW wd 0.01 ≈ Adam vì mô hình chưa quá khớp", optimizer="adamw", lr=lr,
                                 weight_decay=0.01))

def best_of(runs):
    ok = [r for r in runs if r["summary"]["val_macro_f1"] is not None and not r["summary"]["diverged"]]
    return max(ok, key=lambda r: r["summary"]["val_macro_f1"])

BEST_OPT = {"sgd_momentum": best, **{k: best_of(v) for k, v in opt_runs.items()}}
print("\nlr tốt nhất của mỗi bộ (theo val macro-F1):")
for k, r in BEST_OPT.items():
    s = r["summary"]
    print(f"  {k:13s} {r['cfg']['exp_id']:22s} lr={r['cfg']['lr']:<7g} F1 {s['val_macro_f1']:.4f} acc {s['val_acc']:.4f} best_ep {s['best_epoch']}")

ADAM_LR = BEST_OPT["adam"]["cfg"]["lr"]
chk = exp(f"opt-adamw-wd0-lr{ADAM_LR:g}", "optimizer", f"AdamW weight_decay=0, lr={ADAM_LR:g} (kiểm chứng = Adam)",
          "AdamW với wd=0 trùng Adam cùng lr", optimizer="adamw", lr=ADAM_LR, weight_decay=0.0)
ref = BEST_OPT["adam"]
print(f"AdamW wd=0 vs Adam lr={ADAM_LR:g}: F1 {chk['summary']['val_macro_f1']:.6f} vs {ref['summary']['val_macro_f1']:.6f}; "
      f"max |Δ val_loss| theo epoch = {np.max(np.abs(np.array(chk['history']['val_loss']) - np.array(ref['history']['val_loss']))):.2e}")

for k, r in BEST_OPT.items():
    print(verdict(r["cfg"]["exp_id"]))
plot_compare(list(BEST_OPT.values()), ["train_loss", "val_loss", "val_macro_f1"],
             f"{OUT_DIR}/figures/compare_optimizer.png", "Mỗi bộ tối ưu ở lr tốt nhất của nó (theo val)")
display(Image(f"{OUT_DIR}/figures/compare_optimizer.png"))
plot_compare(opt_runs["sgd"] + opt_runs["adam"] + sweep, ["val_macro_f1"],
             f"{OUT_DIR}/figures/compare_optimizer_lr.png", "Độ nhạy với lr: SGD, Adam, SGD+momentum")
display(Image(f"{OUT_DIR}/figures/compare_optimizer_lr.png"))

**Đối chiếu (bộ tối ưu):** xem `REPORT.md` mục 3.2.

### Chủ đề 3 — Hyper-parameter (`hp-*`, cùng với `lr-sgdm-*` ở Part 2)
Mỗi lần đổi một yếu tố của `BASE_CFG`: batch size, độ rộng (`M-wide`), độ sâu (`M-deep`), weight decay, số epoch.

**Dự đoán (trước khi chạy).**
- Batch 128, cùng lr và 20 epoch: gấp 4 số bước cập nhật (2 906 vs 727 bước/epoch) → val F1 cao hơn baseline (vượt 2σ), nhưng thời gian/epoch tăng ~3–4 lần vì GPU xử lý lô nhỏ kém hiệu quả.
- Batch 128 với lr chia 4 (quy tắc tỉ lệ tuyến tính theo chiều ngược lại): *thêm sau lần thăm dò đầu*, khi batch 128 ở lr baseline cho kết quả kém hơn dự đoán. Dự đoán: lr nhỏ hơn bù được nhiễu gradient của lô nhỏ, F1 ít nhất bằng baseline.
- Batch 2048, cùng lr: chỉ 182 bước/epoch → học chậm, F1 thấp hơn rõ. Tăng lr ×4 theo quy tắc tỉ lệ tuyến tính (không khởi động) lấy lại phần lớn khoảng cách nhưng có thể chưa bằng baseline; có nguy cơ dao động ở đầu.
- `M-wide` (161k tham số) và `M-deep` (55,7k): baseline đang chưa khớp (train ≈ val), nên thêm năng lực giúp: wide tăng F1 vượt nhiễu, deep tăng ít hơn.
- Weight decay 1e-4 với SGD: không giúp (không có quá khớp để chữa), khác biệt trong nhiễu.
- 40 epoch: best epoch của baseline là epoch cuối → huấn luyện lâu hơn tăng F1 vượt nhiễu.

In [ ]:
hp_runs = [
    exp("hp-batch128", "hparam", f"batch 128 (lr={BASE_LR:g} giữ nguyên)", "gấp 4 bước/epoch -> F1 cao hơn, chậm hơn/epoch", batch=128),
    exp(f"hp-batch128-lr{BASE_LR/4:g}", "hparam", f"batch 128, lr ÷4 = {BASE_LR/4:g} (quy tắc tỉ lệ tuyến tính)",
        "lr nhỏ bù nhiễu gradient của lô nhỏ; đổi 2 yếu tố có chủ đích (batch + lr)", batch=128, lr=BASE_LR / 4),
    exp("hp-batch2048", "hparam", f"batch 2048 (lr={BASE_LR:g} giữ nguyên)", "chỉ 182 bước/epoch -> F1 thấp hơn rõ", batch=2048),
    exp(f"hp-batch2048-lr{4*BASE_LR:g}", "hparam", f"batch 2048, lr ×4 = {4*BASE_LR:g} (quy tắc tỉ lệ tuyến tính, không warmup)",
        "lr x4 bù phần lớn số bước bị mất; đổi 2 yếu tố có chủ đích (batch + lr)", batch=2048, lr=4 * BASE_LR),
    exp("hp-wide", "hparam", "M-wide 54-512-256-7 (161 287 tham số)", "thêm độ rộng giúp vì baseline chưa khớp", hidden=(512, 256)),
    exp("hp-deep", "hparam", "M-deep 54-256-128-64-7 (55 687 tham số)", "thêm một lớp giúp ít hơn wide", hidden=(256, 128, 64)),
    exp("hp-wd1e-4", "hparam", "weight decay 1e-4 (SGD+momentum)", "không giúp vì chưa quá khớp", weight_decay=1e-4),
    exp("hp-epochs40", "hparam", "40 epoch thay vì 20", "best epoch là cuối -> lâu hơn tăng F1", epochs=40),
]
for r in hp_runs:
    s = r["summary"]
    print(verdict(r["cfg"]["exp_id"]), f"| {s['steps_per_epoch']} bước/epoch, {s['time_per_epoch_s']:.2f}s/epoch")
plot_compare([RESULTS["base-s1"]] + hp_runs[:4], ["val_loss", "val_macro_f1"],
             f"{OUT_DIR}/figures/compare_hparam_batch.png", "Batch size (theo epoch; số bước/epoch khác nhau)")
display(Image(f"{OUT_DIR}/figures/compare_hparam_batch.png"))
plot_compare([RESULTS["base-s1"]] + hp_runs[4:], ["train_loss", "val_loss", "val_macro_f1"],
             f"{OUT_DIR}/figures/compare_hparam_capacity.png", "Độ rộng, độ sâu, weight decay, số epoch")
display(Image(f"{OUT_DIR}/figures/compare_hparam_capacity.png"))

**Đối chiếu (hyper-parameter):** xem `REPORT.md` mục 3.3.

### Chủ đề 4 — Dropout (`drop-*`)
Trước khi chạy, kiểm tra baseline có quá khớp không: khoảng cách `final_val_loss − final_train_loss` của các seed baseline (train loss đo ở `eval()`).

**Dự đoán (trước khi chạy).** Baseline đang chưa khớp (khoảng cách val − train gần 0, cả hai còn giảm), nên dropout — thuốc chữa quá khớp — **không giúp**: q = 0,1 giảm F1 nhẹ (có thể trong nhiễu), q = 0,3 giảm rõ (vượt 2σ) vì mỗi bước chỉ huấn luyện một mạng con, gradient nhiễu hơn, mô hình vốn đã thiếu năng lực lại bị bớt thêm. Khoảng cách val − train vẫn ≈ 0.

In [ ]:
gaps = [r["summary"]["final_val_loss"] - r["summary"]["final_train_loss"] for r in base_runs]
print("baseline: final_val_loss − final_train_loss =", np.round(gaps, 4))
drop_runs = [exp(f"drop-{q:g}", "dropout", f"dropout q={q:g} sau ReLU của 2 lớp ẩn",
                 "baseline chưa khớp nên dropout không giúp; q lớn hại rõ hơn", dropout=q) for q in (0.1, 0.3)]
for r in drop_runs:
    s = r["summary"]
    print(verdict(r["cfg"]["exp_id"]), f"| gap val−train = {s['final_val_loss'] - s['final_train_loss']:+.4f}")
plot_compare([RESULTS["base-s1"]] + drop_runs, ["train_loss", "val_loss", "val_macro_f1"],
             f"{OUT_DIR}/figures/compare_dropout.png", "Dropout (train loss đo ở eval mode)")
display(Image(f"{OUT_DIR}/figures/compare_dropout.png"))

**Đối chiếu (dropout):** xem `REPORT.md` mục 3.4.

### Chủ đề 5 — Gradient clipping (`clip-*`)
`g ← g · min(1, c/‖g‖)`, ‖g‖ là chuẩn L2 toàn cục. Chọn `c` **từ `grad_norm` của baseline**: `c` = trung vị của grad_norm trung bình theo epoch của `base-s1` (làm tròn 2 chữ số có nghĩa), để clipping thật sự kích hoạt ở một phần đáng kể số bước.

**Dự đoán (trước khi chạy).**
- Ở lr baseline: clipping kích hoạt ở khoảng nửa số bước nhưng chỉ thu nhỏ bước một chút (≈ giảm lr hiệu dụng) → F1 trong nhiễu so với baseline.
- Thí nghiệm phản chứng ở lr ×10: không clip → loss nổ thành NaN (bước quá lớn, momentum khuếch đại) hoặc kẹt ở mức rất tệ; có clip → độ dài mỗi bước bị chặn bởi `η·c`, huấn luyện **không** phân kỳ, dù có thể kém baseline. Bằng chứng nằm ở cột `grad_norm` (trước clip) và cờ `diverged`.
- *Thêm sau lần thăm dò đầu:* ở lr ×10, cả hai bản (có/không clip) đều hỏng — lr quá lớn đến mức clip không cứu được. Thêm một cặp ở lr ×3, vùng "dao động nhưng chưa chết", nơi clipping mới có đất dụng võ. Dự đoán: ở lr ×3, bản có clip tốt hơn rõ bản không clip.

In [ ]:
gn_base = np.array(RESULTS["base-s1"]["history"]["grad_norm"])
print("base-s1 grad_norm trung bình theo epoch:", np.round(gn_base, 3))
print("base-s1 grad_norm max theo epoch      :", np.round(RESULTS["base-s1"]["history"]["grad_norm_max"], 3))
C_CLIP = float(f"{np.median(gn_base):.2g}")
MID_LR, HIGH_LR = float(f"{3 * BASE_LR:.2g}"), float(f"{10 * BASE_LR:.2g}")
print(f"c = {C_CLIP:g}; lr cao = {MID_LR:g} (x3) và {HIGH_LR:g} (x10)")
clip_runs = [
    exp(f"clip-{C_CLIP:g}", "clipping", f"clip c={C_CLIP:g} ở lr baseline {BASE_LR:g}",
        "clip kích hoạt ~nửa số bước, F1 trong nhiễu", clip_norm=C_CLIP),
    exp(f"clip-none-lr{MID_LR:g}", "clipping", f"KHÔNG clip ở lr {MID_LR:g} (=3x baseline)",
        "lr x3 không clip -> dao động, kém baseline", lr=MID_LR),
    exp(f"clip-{C_CLIP:g}-lr{MID_LR:g}", "clipping", f"clip c={C_CLIP:g} ở lr {MID_LR:g}",
        "clip chặn các gai gradient -> tốt hơn bản không clip cùng lr", lr=MID_LR, clip_norm=C_CLIP),
    exp(f"clip-none-lr{HIGH_LR:g}", "clipping", f"KHÔNG clip ở lr cao {HIGH_LR:g} (=10x baseline)",
        "lr x10 không clip -> phân kỳ (NaN) hoặc dao động mạnh", lr=HIGH_LR),
    exp(f"clip-{C_CLIP:g}-lr{HIGH_LR:g}", "clipping", f"clip c={C_CLIP:g} ở lr cao {HIGH_LR:g}",
        "clip chặn độ dài bước -> không phân kỳ", lr=HIGH_LR, clip_norm=C_CLIP),
]
for r in clip_runs:
    h = r["history"]
    cf = f"{100*np.mean(h['clip_frac']):.1f}% bước bị clip" if h["clip_frac"] else "-"
    print(verdict(r["cfg"]["exp_id"]), "|", cf, "| diverged =", r["summary"]["diverged"])
plot_compare([RESULTS["base-s1"]] + clip_runs, ["val_loss", "val_macro_f1", "grad_norm_max"],
             f"{OUT_DIR}/figures/compare_clipping.png", "Clipping ở lr baseline và lr x10 (grad_norm đo TRƯỚC clip)")
display(Image(f"{OUT_DIR}/figures/compare_clipping.png"))

**Đối chiếu (clipping):** xem `REPORT.md` mục 3.5.

### Chủ đề 6 — Mixed precision (`amp-*`)
FP16: `autocast(float16)` + `GradScaler` (luôn `unscale_` trước khi đo/clip gradient). BF16: `autocast(bfloat16)`, không GradScaler. Tham số và optimizer vẫn FP32; đánh giá luôn ở FP32. Bộ nhớ ghi là phần cực đại vượt trên dữ liệu nằm sẵn trên GPU.

**Dự đoán (trước khi chạy).** Metric FP16/BF16 trong nhiễu so với FP32. Trên M-base (47k tham số, lô 512) thời gian bị chi phối bởi chi phí gọi kernel chứ không phải phép nhân ma trận, và autocast thêm các phép ép kiểu + GradScaler thêm phép kiểm tra inf → **không nhanh hơn, có thể chậm hơn**. Nếu GPU không có BF16 phần cứng (T4 là Turing 7.5), BF16 sẽ chậm hơn rõ. Với M-wide (161k tham số), phần nhân ma trận lớn hơn nên FP16 có cơ hội nhanh hơn một chút nhờ Tensor Core.

In [ ]:
amp_runs = [exp("amp-fp16", "amp", "FP16 autocast + GradScaler (M-base)", "metric trong nhiễu, không nhanh hơn trên mạng nhỏ", precision="fp16"),
            exp("amp-bf16", "amp", "BF16 autocast, không GradScaler (M-base)",
                "metric trong nhiễu; chậm hơn nếu không có BF16 phần cứng", precision="bf16"),
            exp("amp-fp16-wide", "amp", "FP16 autocast + GradScaler trên M-wide (so với hp-wide)",
                "trên mạng rộng hơn FP16 có thể nhanh hơn chút", precision="fp16", hidden=(512, 256)),
            exp("amp-bf16-wide", "amp", "BF16 autocast trên M-wide (so với hp-wide)",
                "metric trong nhiễu so với hp-wide", precision="bf16", hidden=(512, 256))]
rows = []
for eid in ("base-s1", "amp-fp16", "amp-bf16", "hp-wide", "amp-fp16-wide", "amp-bf16-wide"):
    s = RESULTS[eid]["summary"]
    rows.append(dict(exp_id=eid, precision=RESULTS[eid]["cfg"]["precision"], hidden=str(RESULTS[eid]["cfg"]["hidden"]),
                     s_per_epoch=round(s["time_per_epoch_s"], 3), peak_mem_MB=round(s["peak_mem_MB"], 1),
                     val_acc=round(s["val_acc"], 4), val_macro_f1=round(s["val_macro_f1"], 4)))
print(pd.DataFrame(rows).to_string(index=False))
for eid, ref in (("amp-fp16", "base-s1"), ("amp-bf16", "base-s1"), ("amp-fp16-wide", "hp-wide"), ("amp-bf16-wide", "hp-wide")):
    print(verdict(eid, ref))
plot_compare([RESULTS["base-s1"]] + amp_runs[:2] + [RESULTS["hp-wide"]] + amp_runs[2:], ["val_loss", "val_macro_f1", "epoch_time_s"],
             f"{OUT_DIR}/figures/compare_amp.png", "FP32 vs FP16 vs BF16 (M-base và M-wide)")
display(Image(f"{OUT_DIR}/figures/compare_amp.png"))

**Đối chiếu (mixed precision):** xem `REPORT.md` mục 3.6.

### Chủ đề 7 — Khởi tạo tham số (`init-*`)
Cách đo: `activation_stats` lấy **độ lệch chuẩn sau mỗi `nn.Linear` (tiền kích hoạt, trước ReLU)** trên 4 096 mẫu val ở bước 0; phần tử cuối là std của logits. `xavier` dùng `xavier_normal_` (Var = 2/(n_in+n_out)), không phải 1/n_in của slide. `default` = khởi tạo mặc định của `nn.Linear` (U(±1/√n_in), Var = 1/(3·n_in)).

**Dự đoán (trước khi chạy).**
- `zeros`: mọi nơ-ron trong một lớp giống hệt nhau (đối xứng) và còn tệ hơn: tiền kích hoạt = 0, ReLU(0) = 0, nên `∂L/∂W3 = h2ᵀ·δ = 0`, `∂L/∂W2`, `∂L/∂W1` cũng = 0 vì đi qua `W3 = 0`. Chỉ bias lớp cuối `b3` có gradient → mạng chỉ học được tần suất lớp, đoán luôn lớp đa số: acc ≈ 0,4876, macro-F1 ≈ 0,09. Loss bước 0 đúng bằng ln 7.
- `normal(0; 0,01)`: std tín hiệu nhân ~`0,01·√n_in` ≈ 0,07–0,16 mỗi lớp → logits ≈ 0, loss bước 0 ≈ ln 7 rất sát, gradient lớp đầu rất nhỏ → mấy epoch đầu học chậm hơn He; mạng chỉ 3 lớp nên sau 20 epoch vẫn bắt kịp phần lớn (có thể kém baseline một chút).
- `xavier` ≈ `he` ở mạng 3 lớp (phương sai chỉ khác hệ số ~2 mỗi lớp); `default` có Var nhỏ hơn He 6 lần → tín hiệu co lại ~√6 mỗi lớp, loss bước 0 sát ln 7 hơn, F1 cuối trong nhiễu.
- Loss bước 0 với He cao hơn ln 7 nhiều nhất trong 5 cách vì logits có std lớn nhất.

In [ ]:
rows = []
xb = X_val[:4096]
for init in ("he", "xavier", "default", "normal", "zeros"):
    set_seed(1)
    m_ = MLP(hidden=(256, 128), init=init).to(device)
    st = activation_stats(m_, xb)
    rows.append(dict(init=init, std_L1=round(st[0], 4), std_L2=round(st[1], 4), std_logits=round(st[2], 4),
                     step0_val_loss=round(evaluate(m_, X_val, y_val)["loss"], 4)))
ACT_TABLE = pd.DataFrame(rows)
print("std tiền kích hoạt sau mỗi Linear ở bước 0 (4096 mẫu val), loss bước 0 trên val:")
print(ACT_TABLE.to_string(index=False))

init_runs = [exp(f"init-{k}", "init", f"khởi tạo {k} (thay vì He)", pred, init=k) for k, pred in (
    ("zeros", "đối xứng + ReLU(0)=0 -> chỉ b3 học, đoán lớp đa số, acc≈0.4876"),
    ("normal", "N(0,0.01²): logits≈0, học chậm đầu, gần He sau 20 epoch"),
    ("xavier", "xavier_normal ≈ He ở mạng nông"),
    ("default", "mặc định nn.Linear (Var 1/(3n_in)) ≈ He sau 20 epoch"))]
for r in init_runs:
    print(verdict(r["cfg"]["exp_id"]))

# gradient sau một backward ở bước 0 với init zeros
set_seed(1)
z = MLP(hidden=(256, 128), init="zeros").to(device); z.train()
compute_loss(z(X_val[:512]), y_val[:512], "ce").backward()
print("init zeros — ‖grad‖ từng tham số ở bước 0:",
      {n: round(p.grad.norm().item(), 6) for n, p in z.named_parameters()})
zh = RESULTS["init-zeros"]
zm = build_model(zh["cfg"], device); zm.load_state_dict(zh["best_state"])
print("init-zeros sau huấn luyện: các lớp được dự đoán trên val =", torch.unique(predict(zm, X_val)).tolist(),
      "| ‖W1‖, ‖W2‖, ‖W3‖ =", [round(zm.net[i].weight.norm().item(), 6) for i in (0, 2, 4)],
      "| b3 =", np.round(zm.net[4].bias.detach().cpu().numpy(), 3).tolist())
plot_compare([RESULTS["base-s1"]] + init_runs, ["val_loss", "val_macro_f1", "grad_norm"],
             f"{OUT_DIR}/figures/compare_init.png", "Khởi tạo: He (base-s1) vs xavier / default / normal / zeros")
display(Image(f"{OUT_DIR}/figures/compare_init.png"))

**Đối chiếu (khởi tạo):** xem `REPORT.md` mục 3.7.

In [ ]:
# Tổng hợp mọi lần chạy đến đây (chỉ val)
tab = pd.DataFrame([dict(exp_id=k, group=v["cfg"]["group"], lr=v["cfg"]["lr"], best_epoch=v["summary"]["best_epoch"],
                         best_val_loss=v["summary"]["best_val_loss"], val_acc=v["summary"]["val_acc"],
                         val_macro_f1=v["summary"]["val_macro_f1"], s_per_ep=v["summary"]["time_per_epoch_s"],
                         diverged=v["summary"]["diverged"]) for k, v in RESULTS.items()])
print(tab.sort_values("val_macro_f1", ascending=False).round(4).to_string(index=False))

### Cấu hình cuối cùng (chọn chỉ bằng val)
Nguyên tắc: chỉ đưa vào cấu hình cuối những thay đổi mà **riêng từng cái** đã tăng val macro-F1 vượt 2σ ở Part 3, rồi kiểm tra tổ hợp cũng bằng val. Ứng viên đã thấy ở lần thăm dò: `hp-wide` (thêm độ rộng) và `hp-epochs40` (thêm bước học) — cùng một chẩn đoán: baseline **chưa khớp** (train ≈ val, best epoch ở cuối). Dropout, weight decay, clipping, AMP và các init khác He không vào vì không tăng F1 vượt nhiễu (hoặc làm giảm). Bộ tối ưu: Adam ở lr tốt nhất của nó và SGD+momentum ở lr baseline cách nhau không nhiều, nên để một vòng chọn cuối **trên val** quyết định:

- `final-cand-sgdm`: M-wide + 40 epoch, SGD+momentum, lr baseline.
- `final-cand-adam`: M-wide + 40 epoch, Adam ở lr tốt nhất của Adam (Part 3).

Ứng viên có val macro-F1 cao hơn (seed 1) thành cấu hình cuối, chạy lại 3 seed (`final-s1..3`; `final-s1` lặp lại đúng ứng viên thắng). Ô này không đọc bất cứ thứ gì từ tập eval. Epoch dùng để dự đoán = epoch có val loss thấp nhất của từng lần chạy.

In [ ]:
cands = [run({**BASE_CFG, "exp_id": "final-cand-sgdm", "group": "final", "hidden": (512, 256), "epochs": 40,
              "description": f"Ứng viên cuối: M-wide + 40 epoch, SGD+momentum lr={BASE_LR:g}",
              "notes": "Vòng chọn cuối trên val (seed 1); kết hợp hp-wide + hp-epochs40"}),
         run({**BASE_CFG, "exp_id": "final-cand-adam", "group": "final", "hidden": (512, 256), "epochs": 40,
              "optimizer": "adam", "lr": ADAM_LR,
              "description": f"Ứng viên cuối: M-wide + 40 epoch, Adam lr={ADAM_LR:g}",
              "notes": "Vòng chọn cuối trên val (seed 1); kết hợp hp-wide + hp-epochs40 + Adam lr tốt nhất"})]
win = max(cands, key=lambda r: r["summary"]["val_macro_f1"])
print("ứng viên thắng (val macro-F1):", win["cfg"]["exp_id"])
FINAL_CFG = {k: v for k, v in win["cfg"].items() if k not in ("exp_id", "group", "description", "notes", "seed")}
desc = (f"Cấu hình cuối: M-wide 54-512-256-7, 40 epoch, {win['cfg']['optimizer']} lr={win['cfg']['lr']:g}, "
        f"batch {win['cfg']['batch']}, CE, He, FP32")
final_runs = []
for s in (1, 2, 3):
    final_runs.append(run({**FINAL_CFG, "exp_id": f"final-s{s}", "group": "final", "seed": s, "description": desc + f", seed {s}",
                           "notes": f"Cấu hình cuối, chọn bằng val ({win['cfg']['exp_id']} thắng vòng chọn); đổi nhiều yếu tố có chủ đích"},
                          show=(s == 1)))
ff1 = np.array([r["summary"]["val_macro_f1"] for r in final_runs])
print(f"final val macro-F1 = {ff1.mean():.4f} ± {ff1.std(ddof=1):.4f} (3 seed) | baseline {NOISE['f1_mean']:.4f} ± {NOISE['f1_std']:.4f}")
print(f"Δ = {ff1.mean() - NOISE['f1_mean']:+.4f}  (2σ baseline = {TWO_SIGMA:.4f})")
print("best epoch các seed:", [r["summary"]["best_epoch"] for r in final_runs])
plot_compare(base_runs + cands + final_runs, ["train_loss", "val_loss", "val_macro_f1"],
             f"{OUT_DIR}/figures/compare_final.png", "Baseline (3 seed), 2 ứng viên, cấu hình cuối (3 seed) — chỉ val")
display(Image(f"{OUT_DIR}/figures/compare_final.png"))
SUBMIT = final_runs[0]          # mô hình nộp: final-s1 (seed 1) — chốt TRƯỚC khi nhìn eval
print("Mô hình nộp:", SUBMIT["cfg"]["exp_id"], "seed", SUBMIT["cfg"]["seed"], "best_epoch", SUBMIT["summary"]["best_epoch"])

## Part 4 — Đánh giá cuối trên eval, bảng và nộp bài
Cấu hình đã chốt ở trên trước khi chạy ô này. Dự đoán toàn bộ eval ở `eval()` bằng `best_state` (epoch val loss thấp nhất), ghi `row_id,pred`, rồi chạy **đúng** `scripts/evaluate.py`.
- `predictions_eval.csv` + `eval_result.json` = **final-s1** (file nộp).
- Bằng chứng baseline để riêng: `eval_extra/predictions_eval_base-s1.csv` + `eval_extra/eval_result_base-s1.json`.
- Để có độ nhiễu của điểm eval, các seed còn lại của baseline và cấu hình cuối cũng được chấm (thư mục `eval_extra/`). Không thay đổi gì sau khi thấy các số này.

In [ ]:
os.makedirs(f"{OUT_DIR}/eval_extra", exist_ok=True)
EVAL = {}
EVAL["final-s1"] = final_eval(SUBMIT["cfg"], SUBMIT, data, f"{OUT_DIR}/predictions_eval.csv",
                              repo_root=REPO_ROOT, out_json=f"{OUT_DIR}/eval_result.json")
for r in base_runs + final_runs[1:]:
    eid = r["cfg"]["exp_id"]
    EVAL[eid] = final_eval(r["cfg"], r, data, f"{OUT_DIR}/eval_extra/predictions_eval_{eid}.csv",
                           repo_root=REPO_ROOT, out_json=f"{OUT_DIR}/eval_extra/eval_result_{eid}.json")

In [ ]:
summ = []
for eid in ["base-s1", "base-s2", "base-s3", "final-s1", "final-s2", "final-s3"]:
    summ.append(dict(exp_id=eid, val_macro_f1=round(RESULTS[eid]["summary"]["val_macro_f1"], 4),
                     eval_macro_f1=round(EVAL[eid]["macro_f1"], 4), eval_acc=round(EVAL[eid]["accuracy"], 4)))
S = pd.DataFrame(summ); print(S.to_string(index=False))
be = S[S.exp_id.str.startswith("base")]["eval_macro_f1"]; fe = S[S.exp_id.str.startswith("final")]["eval_macro_f1"]
print(f"eval macro-F1: baseline {be.mean():.4f} ± {be.std(ddof=1):.4f} | final {fe.mean():.4f} ± {fe.std(ddof=1):.4f} | Δ {fe.mean()-be.mean():+.4f}")
print(f"file nộp (final-s1): accuracy {EVAL['final-s1']['accuracy']:.4f}, macro-F1 {EVAL['final-s1']['macro_f1']:.4f}")

# Phân tích lỗi theo lớp từ eval_result.json (file nộp)
with open(f"{OUT_DIR}/eval_result.json") as f:
    ER = json.load(f)
pc = pd.DataFrame(ER["per_class"]).set_index("cls")
names = ["Spruce/Fir", "Lodgepole Pine", "Ponderosa Pine", "Cottonwood/Willow", "Aspen", "Douglas-fir", "Krummholz"]
pc.insert(0, "name", names)
print(pc.round(4).to_string())
cm = np.array(ER["confusion_matrix"])
cmn = cm / cm.sum(1, keepdims=True)
print("\nma trận nhầm lẫn (hàng = thật, cột = dự đoán):"); print(pd.DataFrame(cm).to_string())
hard = int(pc["f1"].idxmin())
off = cmn[hard].copy(); off[hard] = 0
print(f"\nlớp khó nhất: {hard} ({names[hard]}), F1 {pc.loc[hard, 'f1']:.4f}; bị nhầm nhiều nhất sang lớp "
      f"{int(off.argmax())} ({names[int(off.argmax())]}): {100*off.max():.1f}% số mẫu thật của lớp {hard}")
for c in range(7):
    o = cmn[c].copy(); o[c] = 0
    print(f"  lớp {c}: recall {cmn[c, c]:.3f}; nhầm nhiều nhất -> lớp {int(o.argmax())} ({100*o.max():.1f}%)")

fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
for i in range(7):
    for j in range(7):
        ax.text(j, i, f"{cmn[i, j]:.2f}", ha="center", va="center", fontsize=8, color="white" if cmn[i, j] > 0.5 else "black")
ax.set_xlabel("dự đoán"); ax.set_ylabel("nhãn thật"); ax.set_xticks(range(7)); ax.set_yticks(range(7))
ax.set_title("final-s1 trên eval: ma trận nhầm lẫn chuẩn hoá theo hàng (recall)")
fig.colorbar(im); fig.tight_layout(); fig.savefig(f"{OUT_DIR}/figures/compare_final_confusion.png", dpi=110); plt.show()

# Đặc trưng: lớp khó nhất vs lớp hay bị nhầm sang — so mean của các cột số gốc (chưa chuẩn hoá) trên TRAIN
tr = np.load(f"{REPO_ROOT}/data/processed/train.npz"); feat = list(tr["feature_names"])[:10]
cols = [hard, int(off.argmax())]
cmp_ = pd.DataFrame({f"lớp {c} ({names[c]})": tr["X"][tr["y"] == c][:, :10].mean(0) for c in cols}, index=feat)
print("\nmean 10 đặc trưng số trên train:"); print(cmp_.round(1).to_string())

**Phân tích lỗi:** xem `REPORT.md` mục 4.1.

In [ ]:
# Bảng experiments.xlsx từ results/*.json (tạo bằng code, không gõ tay)
results = load_results(f"{OUT_DIR}/results")
order = {"hparam": 1, "baseline": 0, "loss": 2, "optimizer": 3, "dropout": 4, "clipping": 5, "amp": 6, "init": 7, "final": 8}
results.sort(key=lambda r: (order.get(r["cfg"]["group"], 9), r["cfg"]["exp_id"]))
eval_scores = {"base-s1": EVAL["base-s1"], "final-s1": EVAL["final-s1"]}   # eval chỉ cho baseline và cấu hình cuối (bản nộp)
extra = {"base-s1": "eval_* từ eval_extra/eval_result_base-s1.json (scripts/evaluate.py)",
         "final-s1": "MÔ HÌNH NỘP; eval_* từ eval_result.json (scripts/evaluate.py)"}
rows = [to_row(r, eval_scores.get(r["cfg"]["exp_id"]), extra.get(r["cfg"]["exp_id"], "")) for r in results]
assert len(rows) <= 60

def group_note(g):
    rs = [r for r in results if r["cfg"]["group"] == g and r["summary"]["val_macro_f1"] is not None]
    if not rs:
        return ""
    b = max(rs, key=lambda r: r["summary"]["val_macro_f1"]); w = min(rs, key=lambda r: r["summary"]["val_macro_f1"])
    d = b["summary"]["val_macro_f1"] - NOISE["f1_mean"]
    ndiv = sum(r["summary"]["diverged"] for r in results if r["cfg"]["group"] == g)
    return (f"Tốt nhất {b['cfg']['exp_id']} (F1 {b['summary']['val_macro_f1']:.4f}, Δ {d:+.4f} vs TB baseline, "
            f"{'vượt' if abs(d) > TWO_SIGMA else 'trong'} 2σ={TWO_SIGMA:.4f}); kém nhất {w['cfg']['exp_id']} "
            f"(F1 {w['summary']['val_macro_f1']:.4f}); {ndiv} lần phân kỳ. Diễn giải: REPORT.md.")

summary_notes = {g: group_note(g) for g in order}
summary_notes["baseline"] = (f"3 seed: val F1 {NOISE['f1_mean']:.4f} ± {NOISE['f1_std']:.4f}, 2σ = {TWO_SIGMA:.4f}; "
                             f"lr {BASE_LR:g} chọn bằng val (lr-sgdm-*). " + summary_notes["baseline"])
write_xlsx(rows, f"{REPO_ROOT}/templates/experiment_table_template.xlsx", f"{OUT_DIR}/experiments.xlsx",
           seed_ids=["base-s1", "base-s2", "base-s3"], summary_notes=summary_notes)
print(len(rows), "dòng ->", os.path.abspath(f"{OUT_DIR}/experiments.xlsx"))
figs = {f[:-4] for f in os.listdir(f"{OUT_DIR}/figures") if f.endswith(".png") and not f.startswith("compare_")}
assert figs == {r["exp_id"] for r in rows}, figs ^ {r["exp_id"] for r in rows}
print("số ảnh figures/<exp_id>.png =", len(figs), "= số dòng trong bảng")